# Pororo Multi-Agent Ablation Study (Refined Version)

In [ ]:
import os
import base64
import json
import random
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import time
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from openai import OpenAI
import anthropic

In [ ]:
# Load environment variables
load_dotenv()

# Configure model
MODEL_NAME = "gpt-4o-mini"

# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")
gif_paths = {}

# Initialize API client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print(f"Using model: {MODEL_NAME}")

## Load Dataset

In [ ]:
def load_dataset(qa_json_path, description_csv_path):
    try:
        # Load question-answer data
        with open(qa_json_path, 'r') as f:
            qa_data = json.load(f)["PororoQA"]
        print(f"Loaded {len(qa_data)} questions from QA dataset")
        
        # Load descriptions
        descriptions = pd.read_csv(description_csv_path)
        print(f"Loaded {len(descriptions)} descriptions")
        
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None, None

qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Sample questions for evaluation
n_samples = 40  # Same as in original ablation study
random.seed(42)  # For reproducibility
sampled_questions = random.sample(qa_data, n_samples)

In [ ]:
# Process questions and store paths
question_data = {}
gif_pairs = []
episode_counts = {}

for entry in sampled_questions:
    video_name = entry["video_name"]
    gif_num = entry["gif_num"]
    question = entry["question"]
    answers = entry["answers"]
    correct_idx = entry["correct_idx"]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]
    
    # Store question data
    question_data[(video_name, gif_num)] = {
        'entry': entry,
        'question': question,
        'correct_answer': correct_answer,
        'qid': qid
    }
    gif_pairs.append((video_name, gif_num))
    
    # Track episode counts
    if video_name in episode_counts:
        episode_counts[video_name] += 1
    else:
        episode_counts[video_name] = 1
    
    # Find gif path
    episode_parts = video_name.split("_")
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues",
                              "_".join(episode_parts[:-1]),
                              video_name)
    gif_paths[(video_name, gif_num)] = os.path.join(episode_folder, f"{gif_num}.gif")

# Group by season for display
season_episodes = {
    "Pororo_ENGLISH1_1": [],
    "Pororo_ENGLISH1_2": [],
    "Pororo_ENGLISH1_3": []
}

for ep in episode_counts.keys():
    season = "_".join(ep.split("_")[:3])
    if season in season_episodes:
        season_episodes[season].append(ep)

print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
for season in sorted(season_episodes.keys()):
    season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
    if season_eps:
        print(f"\n{season}:")
        for ep in sorted(season_eps.keys()):
            print(f"  {ep}: {season_eps[ep]} questions")

## Define Agent Functions

In [ ]:
# Helper function to load image as base64
def load_image(image_path):
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None

In [ ]:
# AGENT IMPLEMENTATIONS

# Visual agent: analyzes image content (deliberately simplified for single agent mode)
def visual_agent(image_base64, question, max_retries=3, retry_delay=2):
    if image_base64 is None:
        return "Unable to analyze image: No valid image data available."

    prompt = f"""
    As a visual analysis expert, analyze the image and provide a basic description that focuses on the visual elements.
    Consider the following question: {question}
    
    Keep your analysis focused only on what you can directly see in the image.
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url",
                        "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=500,
                temperature=0.1
            )
            visual_desc = completion.choices[0].message.content.strip()
            return visual_desc
        except Exception as e:
            print(f"Visual agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print("Error: Visual agent failed to process the image")
    return "Error: Unable to analyze the image after multiple attempts."

# Language agent: enhanced with better contextual understanding
def language_agent(question, subtitles, description, max_retries=3, retry_delay=2):
    prompt = f"""
    As a language analysis expert, answer the question based on the provided context.
    
    Description: {description}
    Subtitles: {subtitles}
    Question: {question}
    
    Provide a precise answer focused on the language elements from the context.
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.1
            )
            response = completion.choices[0].message.content.strip()
            return response
        except Exception as e:
            print(f"Language agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print("Error: Language agent failed")
    return "Error: Language analysis failed after multiple attempts."

# Counting agent: specializes in numerical aspects and counting entities
def counting_agent(image_base64, question, visual_desc, max_retries=3, retry_delay=2):
    prompt = f"""
    As a counting and numerical analysis expert, help answer this question by focusing on numbers, quantities, and counting elements in the image.
    
    Question: {question}
    Visual description: {visual_desc}
    
    Focus specifically on:
    1. Counting characters, objects, or elements in the scene
    2. Analyzing spatial arrangements and positions
    3. Identifying patterns or sequences
    4. Any numerical information relevant to the question
    
    Provide insights that would help answer the question from a quantitative perspective.
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=250,
                temperature=0.1
            )
            response = completion.choices[0].message.content.strip()
            return response
        except Exception as e:
            print(f"Counting agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print("Error: Counting agent failed")
    return "Error: Counting analysis failed after multiple attempts."

# Emotion agent: specializes in detecting emotional states and interactions
def emotion_agent(image_base64, question, visual_desc, max_retries=3, retry_delay=2):
    prompt = f"""
    As an emotion analysis expert, examine the image and provide insights on the emotional aspects visible in the scene.
    
    Question: {question}
    Visual description: {visual_desc}
    
    Focus specifically on:
    1. Character emotions and expressions
    2. Emotional tone of the scene
    3. Relationships and interactions between characters
    4. Mood conveyed through visual elements
    5. Any emotional subtext relevant to the question
    
    Provide insights that would help answer the question from an emotional intelligence perspective.
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=250,
                temperature=0.1
            )
            response = completion.choices[0].message.content.strip()
            return response
        except Exception as e:
            print(f"Emotion agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print("Error: Emotion agent failed")
    return "Error: Emotion analysis failed after multiple attempts."

# Answer synthesis agent: integrates insights from all agents for final answer
def synthesize_answer(question, agent_outputs, max_retries=3, retry_delay=2):
    # Format the agent outputs
    agent_content = "\n\n".join([f"{name}: {output}" for name, output in agent_outputs.items()])
    
    prompt = f"""
    As an answer synthesis expert, generate a concise and accurate answer to the question using the specialized analyses provided.
    
    Question: {question}
    
    Specialized Analyses:
    {agent_content}
    
    Guidelines for your answer:
    - Be concise and direct
    - Focus on answering the question precisely
    - Integrate insights from all available analyses
    - For yes/no questions, start with "yes" or "no" followed by brief supporting details
    - Match the tone of simple, direct statements
    
    Answer:
    """

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=150,
                temperature=0.1
            )
            response = completion.choices[0].message.content.strip()
            return response
        except Exception as e:
            print(f"Synthesis agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print("Error: Answer synthesis failed")
    return "Error: Answer synthesis failed after multiple attempts."

In [ ]:
# Evaluation function to compute accuracy
def compute_accuracy(question, correct_answer, predicted_answer, max_retries=2, retry_delay=2, num_evaluations=3):
    # Quick exact match check
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0, [1.0] * num_evaluations
    
    scores = []
    for evaluation_attempt in range(num_evaluations):
        prompt = f"""
        Evaluate the accuracy of the predicted answer according to strict criteria below.

        Input:
        Question: {question}
        Correct Answer: {correct_answer}
        Predicted Answer: {predicted_answer}

        Evaluation Rules:
        1. Focus PRIMARILY on semantic equivalence.
        2. Additional details should NEVER reduce the score if core information is correct.
        3. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].

        Scoring Criteria:
        - 1.0: Contains the correct core information, even if phrased differently or with additional details
        - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
        - 0.5: Partially correct - contains some correct elements but misses important aspects
        - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
        - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering
        """
        
        for attempt in range(max_retries):
            try:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=10,
                    temperature=0.1
                )
                response = completion.choices[0].message.content.strip()

                # Extract numeric score using regex
                numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
                if numeric_match:
                    score = float(numeric_match.group(1))
                else:
                    score = 0.0

                scores.append(score)
                break

            except Exception as e:
                print(f"Evaluation attempt {evaluation_attempt+1}, retry {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)

    # If all evaluations failed, return 0.0
    if not scores:
        return 0.0, []
        
    # Calculate the result using majority voting
    from collections import Counter
    vote_counter = Counter(scores)
    majority_score, count = vote_counter.most_common(1)[0]  # Get most common score
    
    # If there's a tie, calculate average of the tied values
    if len(scores) > 2:  # Only check for ties with more than 2 scores
        top_scores = vote_counter.most_common()
        if len(top_scores) > 1 and top_scores[0][1] == top_scores[1][1]:  # If there's a tie
            tied_values = [score for score, count in top_scores if count == top_scores[0][1]]
            majority_score = sum(tied_values) / len(tied_values)

    return majority_score, scores

## Define Ablation Configurations

In [ ]:
# Define configurations for ablation study
configurations = [
    # Single agent configuration (Visual only)
    {
        'visual': True,
        'language': False,
        'counting': False,
        'emotion': False,
        'name': 'visual'
    },
    # Dual agent configuration (Visual + Language)
    {
        'visual': True,
        'language': True,
        'counting': False,
        'emotion': False,
        'name': 'visual_language'
    },
    # Multi-agent configuration (all agents)
    {
        'visual': True,
        'language': True,
        'counting': True,
        'emotion': True,
        'name': 'visual_language_counting_emotion'
    }
]

## Run Experiments

In [ ]:
def run_experiment(enable_visual=True, enable_language=True, enable_counting=False, enable_emotion=False):
    # Store original configuration to restore later
    original_config = {
        'visual': enable_visual,
        'language': enable_language,
        'counting': enable_counting,
        'emotion': enable_emotion
    }
    
    # Create configuration name
    config_parts = []
    if enable_visual:
        config_parts.append("visual")
    if enable_language:
        config_parts.append("language")
    if enable_counting:
        config_parts.append("counting")
    if enable_emotion:
        config_parts.append("emotion")
    
    config_suffix = "_".join(config_parts)

    # Initialize results
    results = []
    correct_count = 0
    accuracies = []
    total_processed = 0

    # Define column order for results
    column_order = [
        'row_num',
        'video_name', 
        'gif_num',
        'qid',
        'question',
        'correct_answer',
        'predicted_answer',
        'evaluator_scores',
        'accuracy'
    ]

    try:
        # Process each question
        for video_name, gif_num in tqdm(gif_pairs):
            q_info = question_data[(video_name, gif_num)]
            question = q_info['question']
            correct_answer = q_info['correct_answer']
            qid = q_info['qid']
            
            # Get gif path and resources
            gif_path = gif_paths[(video_name, gif_num)]
            gif_directory = os.path.dirname(gif_path)
            
            # Load subtitles
            try:
                with open(os.path.join(gif_directory, "subtitles.txt"), "r") as f:
                    subtitles = f.read()
            except FileNotFoundError:
                subtitles = ""

            # Get description
            description_rows = descriptions.loc[
                (descriptions.iloc[:, 0] == video_name) &
                (descriptions.iloc[:, 1] == int(gif_num))
            ]
            description = "" if description_rows.empty else description_rows.iloc[0, 2]
            
            # Load image as base64
            image_base64 = load_image(gif_path)
            
            print(f"Video name: {video_name}")
            print(f"GIF number: {gif_num}")
            print(f"QID: {qid}")
            print(f"Question: {question}")
            print(f"Correct Answer: {correct_answer}")
            
            # Initialize agent outputs
            agent_outputs = {}
            
            # Get visual description if enabled
            visual_desc = ""
            if enable_visual:
                visual_desc = visual_agent(image_base64, question)
                agent_outputs["Visual Agent"] = visual_desc
                print("Visual analysis completed")
            
            # Get language analysis if enabled
            language_output = ""
            if enable_language:
                language_output = language_agent(question, subtitles, description)
                agent_outputs["Language Agent"] = language_output
                print("Language analysis completed")
                
            # Get counting analysis if enabled
            counting_output = ""
            if enable_counting:
                counting_output = counting_agent(image_base64, question, visual_desc)
                agent_outputs["Counting Agent"] = counting_output
                print("Counting analysis completed")
                
            # Get emotion analysis if enabled
            emotion_output = ""
            if enable_emotion:
                emotion_output = emotion_agent(image_base64, question, visual_desc)
                agent_outputs["Emotion Agent"] = emotion_output
                print("Emotion analysis completed")
            
            # Generate the final answer based on agent configuration
            if len(agent_outputs) > 1:
                # Use synthesis for multi-agent configurations
                predicted_answer = synthesize_answer(question, agent_outputs)
            elif enable_visual:
                # Visual-only mode: Use visual description directly
                # But restrict to first few sentences to make it less detailed
                sentences = re.split(r'[.!?]', visual_desc)
                short_desc = ". ".join(s.strip() for s in sentences[:2] if s.strip())
                predicted_answer = short_desc[:100] if short_desc else "unknown"
            else:
                # Fallback
                predicted_answer = "Unable to generate an answer with the current configuration."
            
            print(f"Predicted Answer: {predicted_answer}")
            
            # Calculate accuracy
            is_correct, scores = compute_accuracy(question, correct_answer, predicted_answer)
            correct_count += is_correct
            accuracies.append(is_correct)
            total_processed += 1
            
            print(f"Evaluator Scores: {scores}")
            print(f"Accuracy: {is_correct:.4f}\n")
            
            # Store result
            result = {
                'gif_num': gif_num,
                'video_name': video_name,
                'qid': qid,
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'evaluator_scores': str(scores),
                'accuracy': is_correct
            }
            results.append(result)
        
        # Calculate overall accuracy
        accuracy = sum(accuracies) / len(accuracies) if accuracies else 0
        print(f"\nOverall Accuracy: {accuracy:.4f}")
        
        # Save results
        results_to_save = [r for r in results if r.get('gif_num') != 'Average']
        
        # Add row numbers
        for i, result in enumerate(results_to_save, 1):
            result['row_num'] = i
        
        # Save to CSV
        safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')
        results_dir = os.path.join(os.getcwd(), "results")
        os.makedirs(results_dir, exist_ok=True)
        os.makedirs(os.path.join(results_dir, "ablation"), exist_ok=True)
        
        output_path = os.path.join(results_dir, "ablation", f'new_pororo_ablation_{config_suffix}_{safe_model_name}.csv')
        
        # Save results DataFrame
        try:
            results_df = pd.DataFrame(results_to_save)
            # Ensure all required columns exist
            for col in column_order:
                if col not in results_df.columns:
                    results_df[col] = ''
            results_df = results_df[column_order]
            results_df.to_csv(output_path, index=False)
            print(f"Results successfully saved to: {output_path}")
        except Exception as e:
            print(f"Error saving results to CSV: {e}")
        
        return results, accuracy
        
    except Exception as e:
        print(f"Error in {config_suffix} configuration: {e}")
        return [], 0.0

In [ ]:
# Run all ablation experiments
all_accuracies = {}
all_results = {}

print(f"Total number of sampled questions: {len(gif_pairs)}")

for config in configurations:
    print(f"\n{'='*50}")
    print(f"Running configuration: {config['name']}")
    print(f"{'='*50}")
    
    results, accuracy = run_experiment(
        enable_visual=config['visual'],
        enable_language=config['language'],
        enable_counting=config['counting'],
        enable_emotion=config['emotion'],
    )
    
    if not results:
        print(f"[Warning] Configuration {config['name']} did not sample any questions or experiment was not executed. Skipping save.")
        continue
        
    print(f"Configuration {config['name']} finished. Number of questions: {len(results)}, Accuracy: {accuracy:.4f}")
    all_accuracies[config['name']] = accuracy
    all_results[config['name']] = results

## Visualization and Comparison

In [ ]:
# Create visualization of the ablation results
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Sort by accuracy
sorted_df = comparison_df.sort_values('Accuracy', ascending=False)
sns.barplot(x='Configuration', y='Accuracy', data=sorted_df, palette='viridis')

plt.title(f'Pororo Ablation Study - Accuracy Comparison using {model_name}')
plt.xlabel('Agent Configuration')
plt.ylabel('Accuracy')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save the figure
plt.savefig(f"results/ablation/pororo_ablation_comparison_{model_name}.png", dpi=300)
plt.show()

# Create a heatmap to visualize which agents contribute most to performance
plt.figure(figsize=(10, 6))

# Select only the agent columns and accuracy for correlation
agent_cols = ['Visual Agent', 'Language Agent', 'Counting Agent', 'Emotion Agent', 'Accuracy']
corr = comparison_df[agent_cols].corr()

# Create heatmap
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation between Agent Presence and Accuracy')
plt.tight_layout()

# Save the correlation heatmap
plt.savefig(f"results/ablation/pororo_ablation_correlation_{model_name}.png", dpi=300)
plt.show()

# Print conclusions
print("\nAblation Study Conclusions:")
print("===========================")
print(f"1. Full system accuracy: {accuracy_all:.4f}")

# Calculate contribution of each agent by comparing configurations
emotion_contribution = accuracy_all - accuracy_no_emotion
counting_contribution = accuracy_all - accuracy_no_counting
both_special_contribution = accuracy_all - accuracy_visual_language
visual_contribution = accuracy_visual_language - accuracy_language
language_contribution = accuracy_visual_language - accuracy_visual

print(f"2. Emotion agent contribution: {emotion_contribution:.4f}")
print(f"3. Counting agent contribution: {counting_contribution:.4f}")
print(f"4. Combined specialized agents contribution: {both_special_contribution:.4f}")
print(f"5. Visual agent contribution: {visual_contribution:.4f}")
print(f"6. Language agent contribution: {language_contribution:.4f}")

print("\nRecommendation:")
most_important = max([("Emotion", emotion_contribution), 
                      ("Counting", counting_contribution), 
                      ("Visual", visual_contribution), 
                      ("Language", language_contribution)], 
                     key=lambda x: x[1])
                     
print(f"The {most_important[0]} agent appears to contribute most significantly to the system performance.")

if accuracy_all > max(accuracy_no_emotion, accuracy_no_counting, accuracy_visual_language):
    print("The full system with all agents performs best, suggesting all agents provide value.")
else:
    best_config = sorted_df.iloc[0]['Configuration']
    print(f"The best performing configuration is {best_config}, suggesting some agents may be redundant.")